# Using the DGS Skill

This tutorial explains how to use the **DGS skill** as a natural-language control layer for DeepGeSeq (DGS). The goal is to turn a genomics request into a concrete DGS action: inspect the local package, generate a JSON config, validate it, build the CLI command, and decide when to run a workflow.

**Audience.** DGS users, tutorial authors, and agents that need to operate DeepGeSeq from natural-language instructions.

**Prerequisites.** Basic familiarity with FASTA, BED, BigWig, VCF, PyTorch model checkpoints, and JSON configuration files.

**By the end, you should be able to:**

- Map a natural-language request to a DGS mode: `config`, `run`, `train`, `evaluate`, `explain`, `predict`, or Python API.
- Create a DGS config with `dgs_helper.py`.
- Validate a config before expensive execution.
- Generate the exact DGS CLI command for a reproducible run.
- Know when to use the CLI versus the Python API.


## Tutorial Outline

1. Locate the DeepGeSeq repository and the DGS skill helper.
2. Inspect the local DGS installation and supported models.
3. Translate user intent into DGS workflow modes.
4. Generate and validate a demo JSON config.
5. Build the final CLI command without launching a long run.
6. Review Python API entry points, pitfalls, and extensions.


## What the DGS Skill Does

The DGS skill is a workflow guide for DeepGeSeq. It does not replace DGS itself. Instead, it helps an agent choose the right DGS entry point, collect required files, create a valid configuration, and avoid starting expensive jobs with missing inputs.

Typical requests include:

- "Create a config for training a CNN on FASTA/BED/BigWig data."
- "Run train, evaluate, and predict from one config."
- "Evaluate an existing checkpoint."
- "Explain a trained model with attribution or motif discovery."
- "Predict variant effects from a VCF."
- "Show me the Python API for custom notebook work."


In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys
import textwrap

def find_repo_root(start=None):
    """Find the DeepGeSeq checkout from either the notebook directory or repo root."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "DGS").is_dir() and (candidate / "dgs-skill").is_dir():
            return candidate
    raise RuntimeError("Could not find a DeepGeSeq repo containing DGS/ and dgs-skill/.")

REPO_ROOT = find_repo_root()
EXAMPLE_DIR = REPO_ROOT / "Tutorials" / "0_DGS_usage_example"
HELPER = REPO_ROOT / "dgs-skill" / "scripts" / "dgs_helper.py"
CONFIG_PATH = EXAMPLE_DIR / "dgs_skill_demo_config.json"

print(f"Repository: {REPO_ROOT}")
print(f"Example directory: {EXAMPLE_DIR}")
print(f"DGS helper: {HELPER}")


## Step 1: Inspect the Local DGS Package

The skill starts by checking what DGS package is available in the current checkout. This is useful because model exports, helper behavior, and CLI availability can change across versions.


In [ ]:
def run_helper(*args, cwd=REPO_ROOT, check=True, echo=True):
    """Run dgs_helper.py and return the completed process."""
    cmd = [sys.executable, str(HELPER), *map(str, args)]
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if echo and result.stdout:
        print(result.stdout.rstrip())
    if echo and result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError("Command failed: " + " ".join(shlex.quote(x) for x in cmd))
    return result

inspect_result = run_helper("inspect", "--repo", REPO_ROOT, "--json", echo=False)
dgs_info = json.loads(inspect_result.stdout)

print("DGS version:", dgs_info.get("version"))
print("Package file:", dgs_info.get("package_file"))
print("Console command:", dgs_info.get("console_command") or "not found; use python -m DGS.Main")
print("CLI commands:", ", ".join(dgs_info.get("cli_commands", [])))
print("Models:", ", ".join((dgs_info.get("model_zoo") or dgs_info.get("exported_model_classes") or [])[:20]))


## Step 2: Map Natural Language to a DGS Intent

The skill first decides what the user is actually asking for. This mapping keeps the workflow explicit and helps decide which inputs are mandatory.


In [ ]:
intent_map = {
    "config": "Generate a minimal or full JSON config template.",
    "run": "Execute configured modes in sequence, such as train -> evaluate -> predict.",
    "train": "Train a model from FASTA, intervals, and target tracks.",
    "evaluate": "Evaluate a trained model or checkpoint on held-out intervals.",
    "explain": "Run attribution and motif workflows for a trained model.",
    "predict": "Predict variant effects from a VCF and reference FASTA.",
    "Python API": "Use direct DGS modules for notebooks, custom models, diagnostics, or sequence design.",
}

for intent, meaning in intent_map.items():
    print(f"{intent:10s} -> {meaning}")


## Step 3: Collect Required Inputs

A DGS request should be converted into a small checklist before running anything expensive.

For most train/evaluate workflows, collect:

- `data.genome_path`: reference FASTA.
- `data.intervals_path`: BED-like genomic intervals.
- `data.target_tasks`: one or more BED or BigWig target definitions.
- `model.type` and `model.args.output_size`.
- `train.optimizer`, `train.criterion`, and training limits.

For prediction, also collect:

- `predict.vcf_path`.
- `predict.sequence_length`.

For explanation, also collect:

- `explain.target`.
- attribution method and optional motif output settings.


In [ ]:
request = {
    "intent": ["train", "evaluate", "predict"],
    "genome_path": "Test/reference_grch38p13/GRCh38.p13.genome.fa.gz",
    "intervals_path": "Test/random_regions.bed",
    "targets": [
        "gc_content:bigwig:Test/hg38.gc5Base.bw",
        "recomb:bigwig:Test/recombAvg.bw",
    ],
    "vcf_path": "Test/test.vcf",
    "model": "CNN",
    "device": "cpu",
    "batch_size": 4,
    "max_epochs": 2,
}

print(json.dumps(request, indent=2))


## Step 4: Generate a Demo Config

The helper creates a DGS JSON config from the request. This cell validates schema structure, but it does **not** check whether the FASTA/BED/BigWig/VCF files exist. Turn on `--check-files` only after those files are present.


In [ ]:
args = [
    "new-config",
    "--repo", REPO_ROOT,
    "--example", "full",
    "--mode", "train",
    "--mode", "evaluate",
    "--mode", "predict",
    "--cpu",
    "--output-dir", "dgs_skill_demo_output",
    "--genome", request["genome_path"],
    "--intervals", request["intervals_path"],
    "--target", request["targets"][0],
    "--target", request["targets"][1],
    "--vcf", request["vcf_path"],
    "--model", request["model"],
    "--batch-size", request["batch_size"],
    "--max-epochs", request["max_epochs"],
    "--output", CONFIG_PATH,
    "--validate",
]

run_helper(*args)


## Step 5: Inspect the Config

A quick summary catches common mistakes before execution: wrong mode list, target count mismatch, unexpected device, or missing prediction settings.


In [ ]:
config = json.loads(CONFIG_PATH.read_text())

summary = {
    "modes": config.get("modes"),
    "device": config.get("device"),
    "output_dir": config.get("output_dir"),
    "model": config.get("model", {}).get("type"),
    "output_size": config.get("model", {}).get("args", {}).get("output_size"),
    "target_count": len(config.get("data", {}).get("target_tasks", [])),
    "predict_vcf": config.get("predict", {}).get("vcf_path"),
}

print(json.dumps(summary, indent=2))
assert summary["output_size"] == summary["target_count"], "model.args.output_size must match target task count"


## Step 6: Validate Before Running

Structure validation is quick and should be done before every run. File validation is also recommended, but only when the referenced data files are available.


In [ ]:
run_helper("validate-config", "--repo", REPO_ROOT, "--config", CONFIG_PATH)

missing_files = []
for key in ["genome_path", "intervals_path"]:
    value = config["data"].get(key)
    if value and not (EXAMPLE_DIR / value).exists():
        missing_files.append(value)
for task in config["data"].get("target_tasks", []):
    value = task.get("file_path")
    if value and not (EXAMPLE_DIR / value).exists():
        missing_files.append(value)
vcf_path = config.get("predict", {}).get("vcf_path")
if vcf_path and not (EXAMPLE_DIR / vcf_path).exists():
    missing_files.append(vcf_path)

if missing_files:
    print("File check is intentionally skipped in this lightweight demo.")
    print("Missing example inputs:")
    for item in missing_files:
        print("-", item)
else:
    run_helper("validate-config", "--repo", REPO_ROOT, "--config", CONFIG_PATH, "--check-files")


## Step 7: Build the Execution Command

DGS global runtime flags must appear before the subcommand. For a CPU smoke test, use `--gpu -1`. Use `--no-benchmark` when reproducibility matters.


In [ ]:
command_result = run_helper(
    "command",
    "--command", "run",
    "--config", CONFIG_PATH,
    "--gpu", -1,
    "--seed", 42,
    "--no-benchmark",
    echo=False,
)

command = command_result.stdout.strip()
print(command)


## Step 8: Decide Whether to Execute

The skill should not launch long training or motif-discovery jobs accidentally. First show the command, expected outputs, and any missing data. Then run only when the user explicitly wants execution or when this is a small smoke test.


In [ ]:
RUN_DGS = False

if RUN_DGS:
    # This uses shell=True because the helper may return either
    # "dgs ..." or "python -m DGS.Main ..." depending on the environment.
    subprocess.run(command, cwd=EXAMPLE_DIR, shell=True, check=True)
else:
    expected_outputs = [
        "dgs_skill_demo_output/DGS.log",
        "checkpoints/best_model.pt or checkpoints/final_model.pt",
        "dgs_skill_demo_output/metrics.csv",
        "dgs_skill_demo_output/variant_predictions.csv",
    ]
    print("Execution is disabled for this tutorial cell.")
    print("When real input files are present, set RUN_DGS = True or run the command above.")
    print("Expected outputs:")
    for item in expected_outputs:
        print("-", item)


## CLI Versus Python API

Prefer the DGS CLI for complete reproducible workflows. Prefer the Python API for notebooks, custom models, in-memory diagnostics, profile models, sequence design, or low-level data operations.

Common Python API imports include:

```python
from DGS.Data import build_supervised_dataloaders, build_sequence_dataloader, build_profile_dataloaders
from DGS.DL.Trainer import Trainer
from DGS.DL.Predict import vep_centred_from_files, vep_centred_on_ds
from DGS.DL.Explain import calculate_attributions, motif_enrich
from DGS.DL.Design import gradient_ascent_sequence_design, greedy_ism_sequence_design
from DGS.Model import CNN, DeepSEA, DanQ, Basset, BPNet, ChromBPNet, Borzoi
```

Some of these imports require optional scientific dependencies such as `torch`, `pysam`, or BigWig readers. The next cell checks what is available in the current environment.


In [ ]:
import importlib.util

dependencies = ["torch", "pysam", "pyBigWig", "numpy", "pandas", "sklearn"]
for name in dependencies:
    status = "available" if importlib.util.find_spec(name) else "missing"
    print(f"{name:8s}: {status}")


## Exercise: Turn a Request into a Config

Imagine a user says:

> Train DeepSEA on one binary BED target, evaluate the held-out set, and use CPU for a quick smoke test.

Your task is to fill in the missing file paths and print the helper command that would create the config.


In [ ]:
exercise = {
    "model": "DeepSEA",
    "modes": ["train", "evaluate"],
    "genome_path": "PATH/TO/reference.fa",
    "intervals_path": "PATH/TO/regions.bed",
    "target": "binding:bed:PATH/TO/binding_targets.bed:task_type=binary:target_column=name",
    "output": EXAMPLE_DIR / "exercise_deepsea_config.json",
}

exercise_command = [
    sys.executable, str(HELPER), "new-config",
    "--repo", str(REPO_ROOT),
    "--example", "full",
    "--mode", "train",
    "--mode", "evaluate",
    "--cpu",
    "--model", exercise["model"],
    "--genome", exercise["genome_path"],
    "--intervals", exercise["intervals_path"],
    "--target", exercise["target"],
    "--batch-size", "2",
    "--max-epochs", "1",
    "--output", str(exercise["output"]),
    "--validate",
]

print(" ".join(shlex.quote(str(part)) for part in exercise_command))


## Answer Scaffold

After replacing the placeholder paths with real files, run the printed command. Then validate with `--check-files`, inspect `model.args.output_size`, and generate the final train command.


In [ ]:
answer_steps = [
    "1. Replace PATH/TO/reference.fa, PATH/TO/regions.bed, and PATH/TO/binding_targets.bed.",
    "2. Run the new-config command with --validate.",
    "3. Run validate-config with --check-files once the files exist.",
    "4. Build the command: python dgs_helper.py command --command train --config exercise_deepsea_config.json --gpu -1 --seed 42 --no-benchmark.",
    "5. Start execution only after the config and file checks pass.",
]

print("\n".join(answer_steps))


## Common Pitfalls

- **Global flags in the wrong place.** Use `dgs --gpu -1 run --config config.json`, not `dgs run --gpu -1 --config config.json`.
- **Target count mismatch.** `model.args.output_size` should match the number of scalar target tasks.
- **Skipping file validation.** Use `--check-files` before real training, evaluation, explanation, or prediction.
- **Evaluation-only without a checkpoint.** Provide `evaluate.checkpoint_path` unless evaluation follows training in the same `run`.
- **Optional explain dependencies.** Attribution and motif workflows may require extras such as Captum, tangermeme, and a `modisco` executable.

## Extension

Try changing the config to `modes: ["explain"]`, set `explain.target`, add an evaluation checkpoint, and generate an `explain` command. This is the same skill loop: collect inputs, create or edit config, validate, print command, then execute deliberately.
